# 第 4 章（Google ADK 版）

原书是两个 agent 的**顺序管道**：DraftWriter 写 → FactChecker 查。批评只被打印出来，
**从未回流到生成端**——那是「生成 + 检查」，不是 reflection。

这里补三样东西：

1. **回路**：`LoopAgent` 让两端轮流跑，批评写进 state，下一轮生成端读 `{critique}`
   和自己上一轮的 `{draft_text}` 来**改写**，直到批评端说 `APPROVED`（调用 `exit_loop`）
   或撞上 `max_iterations`。
2. **验收标准放在批评端**：原书两端都没有可检查的条目，批评端于是只能说"挺好"。
   实测把 spec 同时给两端时，第一稿就全满足、直接 APPROVED，回路空转；
   所以这里按真实分工来——生成端只拿 brief，批评端拿 spec（相当于「测试套件」）。
3. **两种停法都要能看见**：`exit_loop` 提前结束，和 `max_iterations` 硬停。

kernel 必须选 **`Python (agentic-adk)`**。

本次运行的实际轨迹（**3 轮才收敛，中间出现过一次回归**：第 2 稿修好了缺英尺和
缺批评者，却把设计师名字改丢了，被批评端抓回来）和两个 ADK 特有的坑，都记在 `../../ch04_reflection/README.md` 第十二节。


In [1]:
import re
import time
import uuid

from google.adk.agents import LlmAgent, LoopAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import exit_loop
from google.genai import types

# 【适配】原书 gemini-2.0-flash 已下线。
#   免费层是「每个模型独立 20 次/天」，所以 gemini-3.5-flash 用完时可以换桶：
#   实测还有额度的有 gemini-3-flash-preview / gemini-3.1-flash-lite。
MODEL = "gemini-3-flash-preview"

SUBJECT = "The Eiffel Tower"

# 验收标准只给批评端 —— 相当于「作者按 brief 写，评审按 spec 查」。
# 真实系统里这份 spec 就是测试套件 / rubric / linter。
SPEC = """
    1. name the engineer who designed it;
    2. state the year it was completed;
    3. give the height in BOTH metres and feet;
    4. mention who specifically criticised it when it was built (not just "many people").
    Keep it to 3-5 sentences, one paragraph.
"""

# ★ 补充：生成端读 {critique} 和 {draft_text}（自己上一轮的稿子），所以第 2 轮起是【改写】
#   而不是重新写一篇。原书这里没有这两个占位符，也就没有回路的入口 —— 这就是
#   「批评从未用于重新生成」的字面含义。
draft_writer = LlmAgent(
    name="DraftWriter",
    model=MODEL,
    description="Writes the first draft, then revises it according to the critique.",
    instruction="""
    You write a short, informative paragraph (3-5 sentences) about: {subject}

    Your previous draft (empty on the first pass):
    {draft_text}

    Critique of that draft (empty on the first pass):
    {critique}

    If a critique is present this is a REVISION: rewrite the paragraph so that every point the
    critique raises is fixed, while keeping whatever was already correct.
    If no critique is present, write the first draft.
    Output only the paragraph itself — no preamble, no headings.
    """,
    output_key="draft_text",
)

# ★ 补充：批评端必须能【喊停】。有了 exit_loop，它批准时可以直接结束循环；
#   没有这个工具，循环只能靠 max_iterations 硬停。
fact_checker = LlmAgent(
    name="FactChecker",
    model=MODEL,
    description="Checks the draft against the spec and either approves it or lists problems.",
    instruction=f"""
    You are a meticulous fact-checker. Check this draft against the requirements:

    {SPEC}

    Draft:
    {{draft_text}}

    Respond with EXACTLY one of:
    - the single word APPROVED — only if every requirement is concretely satisfied, AND call
      the exit_loop tool in the same turn;
    - otherwise a bulleted list of concrete problems (at most 4 bullets), each naming the
      requirement number that fails and quoting the offending part.
      Do not rewrite the draft yourself.
    """,
    output_key="critique",
    tools=[exit_loop],
)

# ★ 补充：LoopAgent 才是「反思」的载体 —— 一个 loop 里两个 agent 轮流跑，
#   state 是它们之间的唯一通道（Ch3 已经见过：output_key 写、{占位符} 读）。
#   max_iterations 是预算保险丝：批评端一直不批准时，别无限烧 token。
review_loop = LoopAgent(
    name="DraftCritiqueLoop",
    sub_agents=[draft_writer, fact_checker],
    max_iterations=3,
)


runner = InMemoryRunner(review_loop)
USER_ID = "user_1"


def _text(content) -> str:
    """【适配】Content 没有 .text 属性，必须走 parts。"""
    if getattr(content, "text", None):
        return content.text
    return "".join(p.text for p in (getattr(content, "parts", None) or [])
                   if getattr(p, "text", None))


def _retry_delay(msg: str, default: int = 30) -> int:
    m = re.search(r"retryDelay['\"]?\s*:\s*['\"]?(\d+(?:\.\d+)?)s", msg)
    return min(int(float(m.group(1))) + 5, 180) if m else default


def run_reflection(subject: str, max_retry: int = 2):
    """返回 (最终稿, 最后一次批评, 回合数)。回合数 0 表示这次没跑完。"""
    for attempt in range(1, max_retry + 2):
        session_id = str(uuid.uuid4())
        runner.session_service.create_session_sync(
            app_name=runner.app_name, user_id=USER_ID, session_id=session_id,
            # 【适配】三个键都先给空串：ADK 只替换 {占位符}，state 里缺 key 就填不进去，
            #   而第一轮运行时 critique / draft_text 本来就还不存在。
            state={"subject": subject, "draft_text": "", "critique": ""},
        )
        try:
            step = 0
            for event in runner.run(
                user_id=USER_ID,
                session_id=session_id,
                new_message=types.Content(role="user", parts=[types.Part(text=subject)]),
            ):
                if event.is_final_response() and event.content:
                    step += 1
                    body = _text(event.content).strip()
                    print(f"\n--- [{step}] {getattr(event, 'author', '?')} ---")
                    # 调用 exit_loop 的那一轮，skip_summarization 会把文本抑制成空，
                    # 真正的内容在 state 里（下面统一读出来）。
                    print(body or "(空 —— 这一轮调了 exit_loop，文本被跳过；内容见 state)")

            session = runner.session_service.get_session_sync(
                app_name=runner.app_name, user_id=USER_ID, session_id=session_id
            )
            return session.state.get("draft_text", ""), session.state.get("critique", ""), step

        except Exception as e:
            msg = str(e)
            head = (msg.strip().splitlines() or [type(e).__name__])[0][:160]

            # ⚠️ 分清两种 429：PerMinute 是限流（等一等就好），PerDay 是当天用完了（等也没用）。
            if "PerDay" in msg:
                print(f"   ⛔ {MODEL} 的【每日】配额用完了（免费层 20 次/天/模型）—— 重试没意义。")
                print("      换一个模型桶（每个模型独立计数），或等配额重置。")
                return "", "", 0

            transient = any(k in msg for k in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE"))
            if transient and attempt <= max_retry:
                wait = _retry_delay(msg)
                print(f"   ⏳ 第 {attempt} 次撞瞬时错误，等 {wait}s 后重试")
                time.sleep(wait)
                continue

            print(f"   ❌ {type(e).__name__}: {head}")
            return "", "", 0

    return "", "", 0


draft, critique, steps = run_reflection(SUBJECT)

print("\n" + "=" * 26 + " 结果 " + "=" * 26)
if steps == 0:
    print("\n⚠️ 这次没跑完（看上面的错误行）。下面没有结果可看。")
else:
    print(f"\n共 {steps} 个 agent 回合 = 草稿数 + 批评数")
    print("\n--- 最终稿（session.state['draft_text']）---")
    print(draft.strip() or "(空)")
    print("\n--- 最后一次批评（session.state['critique']）---")
    print(critique.strip() or "(空)")
    print()
    if "APPROVED" in critique.upper():
        print("✅ 批评端批准了，循环靠 exit_loop 提前结束")
    else:
        print("⚠️ 批评端始终没批准，循环是被 max_iterations 硬停的 —— 这正是「预算保险丝」在起作用")

/tmp/ipykernel_3087370/2141920279.py:79: DeprecationWarning: LoopAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  review_loop = LoopAgent(
Deprecated. Please migrate to the async method.



--- [1] DraftWriter ---
The Eiffel Tower is an iconic iron lattice structure located on the Champ de Mars in Paris, France. Named after engineer Gustave Eiffel, it was completed in 1889 as the grand centerpiece for the World's Fair and to commemorate the centennial of the French Revolution. Standing at 330 meters tall, it held the title of the world's tallest man-made structure for 41 years until the Chrysler Building was finished in 1930. Today, it remains a global cultural landmark and one of the most visited paid monuments in the world, symbolizing both industrial innovation and French heritage.



--- [2] FactChecker ---
* Requirement 3 fails: The height is provided only in meters ("330 meters tall") and lacks the corresponding measurement in feet.
* Requirement 4 fails: The draft does not mention who specifically criticised the tower when it was built, omitting this detail entirely.



--- [3] DraftWriter ---
The Eiffel Tower is an iconic iron lattice structure in Paris, France, completed in 1889 as the centerpiece for the World's Fair commemorating the centennial of the French Revolution. Standing at 330 meters (1,083 feet) tall, it remained the world’s tallest man-made structure for 41 years until the Chrysler Building was finished in 1930. During its construction, the tower was famously criticized by a group of prominent artists and intellectuals, including writer Guy de Maupassant, who dismissed the structure as a "gigantic black smokestack." Despite this early opposition, it has become a global cultural landmark and remains one of the most visited paid monuments in the world.



--- [4] FactChecker ---
* Requirement 1 fails: The draft does not name the engineer who designed the tower, referring only to the monument's name in the opening sentence ("The Eiffel Tower is an iconic iron lattice structure...").



--- [5] DraftWriter ---
Named after its designer, Gustave Eiffel, the Eiffel Tower is an iconic iron lattice structure in Paris completed in 1889 as the centerpiece for the World's Fair. Standing at 330 meters (1,083 feet) tall, it remained the world’s tallest man-made structure for 41 years until the Chrysler Building was finished in 1930. During its construction, the tower was famously criticized by a group of prominent artists and intellectuals, including writer Guy de Maupassant, who dismissed the structure as a "gigantic black smokestack." Today, it is recognized as a global cultural landmark and remains one of the most visited paid monuments in the world.


Deprecated. Please migrate to the async method.



--- [6] FactChecker ---
(空 —— 这一轮调了 exit_loop，文本被跳过；内容见 state)

========================== 结果 ==========================

共 6 个 agent 回合 = 草稿数 + 批评数

--- 最终稿（session.state['draft_text']）---
Named after its designer, Gustave Eiffel, the Eiffel Tower is an iconic iron lattice structure in Paris completed in 1889 as the centerpiece for the World's Fair. Standing at 330 meters (1,083 feet) tall, it remained the world’s tallest man-made structure for 41 years until the Chrysler Building was finished in 1930. During its construction, the tower was famously criticized by a group of prominent artists and intellectuals, including writer Guy de Maupassant, who dismissed the structure as a "gigantic black smokestack." Today, it is recognized as a global cultural landmark and remains one of the most visited paid monuments in the world.

--- 最后一次批评（session.state['critique']）---
APPROVED

✅ 批评端批准了，循环靠 exit_loop 提前结束
